# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Frederic7/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis.** One row = one pseudonymized content page, uniquely identified by
`content_id`. Each page belongs to one `client_id` pseudonym. The starter slice contains
30,000 pages across 32 clients. Verified below: 0 duplicate `content_id` values in the
30,000 rows.

**Time window.** All metrics on a row are aggregated over a single trailing 90-day window
ending at the CSV's export snapshot date (the exact date is not encoded in the file — see
Section 4, Data limits). Columns are grouped into three sub-windows:

- `*_90d` (e.g. `impressions_90d`, `clicks_90d`): totals over the full 90-day window.
- `*_last_30d`: days 0–30 back from the snapshot (the most recent 30 days).
- `*_prev_30d`: days 31–60 back (the 30-day period before `*_last_30d`).

`trend_pct` and `trend_direction` compare `*_last_30d` to `*_prev_30d` impressions. Days
61–90 back contribute to `*_90d` totals but have no dedicated comparison column.


> Measurement note: `impressions_*` and `days_with_impressions` come from Google Search
> Console (organic search only). `sessions_*`, `pageviews_*`, `users_*`, and
> `days_with_sessions` come from Google Analytics 4 (all traffic channels). So a page can
> legitimately have sessions on a day with no GSC impressions (direct, referral, social, or
> AI-referred traffic). See Section 4 for cross-system limits.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1 code: grain probe, row counts, window-column inventory
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Portable repo-root walk-up (works from repo root OR work/notebooks/) ---
NB_PATH = Path(os.path.abspath('')).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / 'data' / 'raw').is_dir() and (REPO_ROOT / 'scripts' / 'ml_utils.py').is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(
        'Could not locate repo root. Expected a folder with data/raw/ and '
        f'scripts/ml_utils.py. Search started from: {NB_PATH}'
    )
SCRIPTS_DIR = str(REPO_ROOT / 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
assert RAW_PATH.is_file(), f'Missing starter CSV: {RAW_PATH}'
print(f'Repo root: {REPO_ROOT}')
print(f'Starter CSV: {RAW_PATH.stat().st_size / 1024 / 1024:.1f} MB')

# --- Load ---
df = pd.read_csv(RAW_PATH)

# --- 1. Grain probe: content_id should be unique (0 dupes means 1 row = 1 page) ---
total_rows = len(df)
unique_content_ids = df['content_id'].nunique()
dup_check = (
    df.groupby('content_id', dropna=False)
      .size()
      .rename('rows_per_id')
      .reset_index()
      .query('rows_per_id > 1')
)
print('\n=== GRAIN ===')
print(f'Total rows            : {total_rows:,}')
print(f'Unique content_id     : {unique_content_ids:,}')
print(f'Duplicate content_ids : {len(dup_check)}  (expect 0)')
print(f'Unique client_id      : {df["client_id"].nunique()}  (expect 32)')

# --- 2. Window column inventory ---
cols = list(df.columns)
cols_90d = [c for c in cols if c.endswith('_90d')]
cols_last30 = [c for c in cols if c.endswith('_last_30d')]
cols_prev30 = [c for c in cols if c.endswith('_prev_30d')]
print('\n=== WINDOW COLUMNS ===')
print(f'*_90d       ({len(cols_90d)}):  {cols_90d}')
print(f'*_last_30d  ({len(cols_last30)}):  {cols_last30}')
print(f'*_prev_30d  ({len(cols_prev30)}):  {cols_prev30}')

# --- 3. Range checks that confirm the 90-day window story ---
print('\n=== WINDOW SANITY (expect min/max bounded by 90-day semantics) ===')
print(f'content_age_days         min={df["content_age_days"].min():.0f}  median={df["content_age_days"].median():.0f}  max={df["content_age_days"].max():.0f}  (all >= 90 in this slice)')
print(f'days_with_impressions    min={df["days_with_impressions"].min():.0f}  median={df["days_with_impressions"].median():.0f}  max={df["days_with_impressions"].max():.0f}  (0-90 range expected)')
print(f'days_with_sessions       min={df["days_with_sessions"].min():.0f}  median={df["days_with_sessions"].median():.0f}  max={df["days_with_sessions"].max():.0f}  (0-90 range expected)')
print(f'impressions_90d          min={df["impressions_90d"].min():.0f}  (> 0 expected; prep filter requires it)')

# --- 4. Cross-system gap: days_with_sessions can exceed days_with_impressions ---
gap = df['days_with_sessions'] - df['days_with_impressions']
n_gap_positive = int((gap > 0).sum())
n_gap_zero = int((gap == 0).sum())
n_gap_negative = int((gap < 0).sum())
print('\n=== CROSS-SYSTEM: days_with_sessions vs days_with_impressions ===')
print(f'Rows with sessions_days > impressions_days : {n_gap_positive:,}  ({n_gap_positive/total_rows*100:.1f}%)')
print(f'Rows with sessions_days = impressions_days : {n_gap_zero:,}  ({n_gap_zero/total_rows*100:.1f}%)')
print(f'Rows with sessions_days < impressions_days : {n_gap_negative:,}  ({n_gap_negative/total_rows*100:.1f}%)')
if n_gap_positive:
    print(f'  Positive gap (extra session-days): min={gap[gap>0].min()}, median={gap[gap>0].median():.1f}, max={gap[gap>0].max()}')
if n_gap_negative:
    print(f'  Negative gap (extra impression-days): min={gap[gap<0].min()} (most negative), median={gap[gap<0].median():.1f}')

Repo root: /Users/frederic/Desktop/git-repo/flyrank-internship-ml
Starter CSV: 6.4 MB

=== GRAIN ===
Total rows            : 30,000
Unique content_id     : 30,000
Duplicate content_ids : 0  (expect 0)
Unique client_id      : 32  (expect 32)

=== WINDOW COLUMNS ===
*_90d       (8):  ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']
*_last_30d  (3):  ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']
*_prev_30d  (3):  ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

=== WINDOW SANITY (expect min/max bounded by 90-day semantics) ===
content_age_days         min=90  median=236  max=564  (all >= 90 in this slice)
days_with_impressions    min=1  median=81  max=88  (0-90 range expected)
days_with_sessions       min=1  median=6  max=90  (0-90 range expected)
impressions_90d          min=1  (> 0 expected; prep filter requires it)

=== CROSS-SYSTEM: days_with_sessions vs d

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Every field the pipeline touches (44 raw CSV columns + 8 columns added by `scripts/01_prepare_features.py`), sorted into four buckets. Excluded fields each carry a one-line why.

### Context (grouping / splitting / reading only; never model-learned)

- `content_id` — pseudonymous page key; unique row grain; for joins and grouping, never a feature.
- `client_id` — pseudonymous client key; 32 values in the starter slice. Used for client-holdout train/test splits.

### Label / proxy (y train/test splits only).

Only used to create `y` or its direct inputs. **Never a feature.

- `trend_direction` — Label precursor; `trend_pct` — raw label input to `trend_direction`.  `_declining_label` — **The target.** 1 = `trend_direction == "down"`. **impressions_last_30d** — Direct numerator input `trend_pct` formula.
- `impressions_prev_30d` — Direct denominator input to `trend_pct` formula.
- `clicks_last_30d`, `sessions_last_30d` — Same 30-day window as the label's numerator. Contemporaneous with the outcome being labeled — leakage-adjacent; excluded per the label-trap rule.
- `clicks_prev_30d`, `sessions_prev_30d` — Same 30-day window as the label's denominator. Mirror same logic as `impressions_prev_30d; excluded from features per window-alignment rule.

### Features (knowable at prediction moment; exactly the contents of `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` in ml_utils.py)

**Numeric features (19 columns):**
- Keyword context: `search_volume`, `competition`, `cpc`
- Content properties: `word_count`, `char_count`, `content_age_days`, `days_since_last_update`
- Derived 90-day (logged totals (logged):  `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`
- 90-day activity counts: `days_with_impressions`, `days_with_sessions`
- Derived rates (×100 percentages): `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- Position: `avg_position` (0 = no data, handled downstream)

**Categorical features (9 columns):**
- Keyword context: `competition_level`, `content_type`, `main_intent`
- Property tiers: `age_tier`, `freshness_tier`, `word_count_tier`
- Performance tiers: `impression_tier`, `position_tier`

### Excluded (why)

Each field is excluded from features AND labels (because it leaks, encodes production context, or is a duplicate-derived signal already captured above:

- `provider_used` — Production metadata about which LLM vendor generated the article; a product decision, not a user signal. Data dictionary marks it "Not a model feature."
- `model_used` — LLM model name; same reason as `provider_used`.
- `char_count_tier` — Duplicate signal to `word_count_tier` (correlated near 1.0); drop one to avoid multicollinearity. Model categorical features exclude it.
- `impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`, `engaged_sessions_90d`, `pageviews_90d`, `users_90d`, `scroll_events_90d` — Raw 90-day totals are replaced by their `log_*` transforms in the model (traffic;logged variants go into features; `engaged_sessions_90d`, `pageviews_90d`, `users_90d`, `scroll_events_90d` excluded because derived rates `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` already capture the ratio — raw totals double-count.
- `age_tier_order` — Numeric ordinal duplicate of the categorical `age_tier`; the model uses the categorical.
- `has_clicks`, `has_ai_sessions` — Binary flags of already captured by the log-transforms (log1p(0)=0 distinguishes zero from non-zero implicitly; excluded to avoid duplicate signal.
- `measurable_opportunity` — Downstream filter flag, not a predictive signal. Excluded so the model does not learn a rule threshold by content-type-specifically:  `users_90d` — Used by rule filter; excluded to avoid duplicate signal with `sessions_90d` and rate columns.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 2 code: verify field buckets are complete, no overlaps, matches pipeline
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Portable repo-root walk-up ---
NB_PATH = Path(os.path.abspath('')).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / 'data' / 'raw').is_dir() and (REPO_ROOT / 'scripts' / 'ml_utils.py').is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(f'Repo root not found from {NB_PATH}')
SCRIPTS_DIR = str(REPO_ROOT / 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
assert RAW_PATH.is_file(), f'Missing CSV: {RAW_PATH}'

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

# --- Load raw CSV + add 8 prep-script columns ---
df = pd.read_csv(RAW_PATH)
prep_cols_from_script = [
    # mimic the prep script's added columns for bucket check
]
# Build the full 52-column universe
# 44 raw + 8 prep-added
prep_added = [
    'is_declining_label',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
]

# --- Define our four buckets (match the markdown above ---
CONTEXT = ['content_id', 'client_id']
LABEL_PROXY = [
    'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'impressions_prev_30d',
    'clicks_last_30d', 'sessions_last_30d',
    'clicks_prev_30d', 'sessions_prev_30d',
]
FEATURES_NUMERIC = list(MODEL_NUMERIC_FEATURES)  # 19
FEATURES_CATEGORICAL = list(MODEL_CATEGORICAL_FEATURES)  # 9
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL  # 28 total
EXCLUDED = [
    'provider_used', 'model_used', 'char_count_tier',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'engaged_sessions_90d', 'pageviews_90d', 'users_90d', 'scroll_events_90d',
    'age_tier_order', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
]

# --- Check 1: every raw columns in exactly one bucket ---
all_buckets = {
    'Context': CONTEXT,
    'Label / proxy': LABEL_PROXY,
    'Features': FEATURES,
    'Excluded': EXCLUDED,
}

all_assigned = []
for bucket_name, cols in all_buckets.items():
    for c in cols:
        all_assigned.append(c)
dupes = [c for c in set(all_assigned) if all_assigned.count(c) > 1]
print('=== BUCKET CHECK 1: mutual exclusivity (expect 0 duplicates)')
print(f'  Duplicate assignments: {dupes if dupes else "(none — good)"}')

# --- Check 2: union coverage (44 raw + 8 prep = 52 columns ---
all_raw_cols = list(df.columns)
all_full_universe = all_raw_cols + prep_added
missing_prep = [c for c in prep_added if c not in all_assigned]
missing_raw = [c for c in all_raw_cols if c not in all_assigned]
extra_assigned = [c for c in all_assigned if c not in all_full_universe]
print(f'\n=== BUCKET CHECK 2: coverage')
print(f'  Raw CSV cols        : {len(all_raw_cols)} (expect 44)')
print(f'  Prep-added cols      : {len(prep_added)} (expect 8)')
print(f'  Total universe       : {len(all_full_universe)} (expect 52)')
print(f'  Total assigned     : {len(all_assigned)} (expect 52)')
print(f'  Raw cols unassigned: {missing_raw if missing_raw else "(none — good)"}')
print(f'  Prep cols unassigned: {missing_prep if missing_prep else "(none — good)"}')
print(f'  Extra (not in universe): {extra_assigned if extra_assigned else "(none — good)"}')

# --- Check 3: feature lists match ml_utils.py exactly ---
print(f'\n=== BUCKET CHECK 3: match ml_utils.MODEL_*_FEATURES')
num_match_numeric = sorted(FEATURES_NUMERIC) == sorted(MODEL_NUMERIC_FEATURES)
num_match_categorical = sorted(FEATURES_CATEGORICAL) == sorted(MODEL_CATEGORICAL_FEATURES)
print(f'  Numeric features match : {num_match_numeric} (19 expected)')
print(f'  Categorical features match: {num_match_categorical} (9 expected)')

# --- Check 4: counts per bucket + summary table ---
print(f'\n=== BUCKET SUMMARY')
summary = pd.DataFrame([
    {'Bucket': name, 'N columns': len(cols), 'Examples': ', '.join(cols[:4]) + ('...' if len(cols) > 4 else '')}
    for name, cols in all_buckets.items()
])
print(summary.to_string(index=False))

=== BUCKET CHECK 1: mutual exclusivity (expect 0 duplicates)
  Duplicate assignments: (none — good)

=== BUCKET CHECK 2: coverage
  Raw CSV cols        : 44 (expect 44)
  Prep-added cols      : 8 (expect 8)
  Total universe       : 52 (expect 52)
  Total assigned     : 52 (expect 52)
  Raw cols unassigned: (none — good)
  Prep cols unassigned: (none — good)
  Extra (not in universe): (none — good)

=== BUCKET CHECK 3: match ml_utils.MODEL_*_FEATURES
  Numeric features match : True (19 expected)
  Categorical features match: True (9 expected)

=== BUCKET SUMMARY
       Bucket  N columns                                                                Examples
      Context          2                                                   content_id, client_id
Label / proxy          9 trend_direction, trend_pct, is_declining_label, impressions_last_30d...
     Features         26                          search_volume, competition, cpc, word_count...
     Excluded         15          provider_u

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


Five checks back the claims in Sections 1 and 2:

1. **Grain hold.** No duplicate `content_id` values (one row = one page).
2. **Counts match docs.** 30,000 rows, 32 clients, 44 raw columns + 8 prep-added = 52 total.
3. **Window ranges make sense.** `days_with_impressions` and `days_with_sessions` live in 0–90; cross-system gap between them quantified.
4. **Field buckets are clean.** 52 columns assigned; no overlap; feature list matches `ml_utils.MODEL_*_FEATURES`.
5. **Missingness is patterned.** Not random — keyword-data blanks follow `content_type` (the flyrank-data skill's §24 warning).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3 code: verify every claim above (grain, counts, windows, buckets, missingness)
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Portable repo-root walk-up ---
NB_PATH = Path(os.path.abspath('')).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / 'data' / 'raw').is_dir() and (REPO_ROOT / 'scripts' / 'ml_utils.py').is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(f'Repo root not found from {NB_PATH}')
SCRIPTS_DIR = str(REPO_ROOT / 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
assert RAW_PATH.is_file(), f'Missing CSV: {RAW_PATH}'

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

df = pd.read_csv(RAW_PATH)
TOTAL_ROWS = len(df)
prep_added = [
    'is_declining_label','log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d',
    'has_clicks','has_ai_sessions','measurable_opportunity',
]
# Build the 52-column universe
all_raw_cols = list(df.columns)
all_universe = all_raw_cols + prep_added

# --- Block 1: GRAIN (Section 1 claim: one row = one content_id) ---
print('=' * 70)
print('CHECK 1 / 5 — GRAIN: one row = one content_id')
print('=' * 70)
dupes = df.groupby('content_id').size().reset_index(name='n').query('n > 1')
uniq_clients = df['client_id'].nunique()
print(f'Total rows               : {TOTAL_ROWS:,}  (expect 30,000)')
print(f'Unique content_id        : {df["content_id"].nunique():,}  (expect {TOTAL_ROWS:,})')
print(f'Duplicate content_ids    : {len(dupes)}  (expect 0 — grain holds)')
print(f'Unique client_id         : {uniq_clients}  (expect 32)')
grain_ok = (len(dupes) == 0) and (df['content_id'].nunique() == TOTAL_ROWS)
print(f'Grain holds?             : {grain_ok}')

# --- Block 2: COUNTS match docs ---
print('\n' + '=' * 70)
print('CHECK 2 / 5 — COUNTS (columns, rows, clients)')
print('=' * 70)
print(f'Raw CSV cols             : {len(all_raw_cols)}  (expect 44)')
print(f'Prep-script added cols   : {len(prep_added)}  (expect 8)')
print(f'Total contract universe  : {len(all_universe)}  (expect 52)')
# Client distribution (quick sanity: minimum 1 client with fewest pages)
client_counts = df['client_id'].value_counts()
print(f'Pages per client — min={client_counts.min():,}, median={client_counts.median():.0f}, max={client_counts.max():,}')

# --- Block 3: WINDOW ranges ---
print('\n' + '=' * 70)
print('CHECK 3 / 5 — WINDOW RANGES (must fit 90-day semantics)')
print('=' * 70)
print(f'content_age_days     min={df["content_age_days"].min():.0f}   median={df["content_age_days"].median():.0f}   max={df["content_age_days"].max():.0f}')
print(f'days_with_impressions min={df["days_with_impressions"].min():.0f}   median={df["days_with_impressions"].median():.0f}   max={df["days_with_impressions"].max():.0f}  (expect 0-90)')
print(f'days_with_sessions    min={df["days_with_sessions"].min():.0f}   median={df["days_with_sessions"].median():.0f}   max={df["days_with_sessions"].max():.0f}  (expect 0-90)')
gap = df['days_with_sessions'] - df['days_with_impressions']
pos_gap = (gap > 0).sum()
neg_gap = (gap < 0).sum()
zero_gap = (gap == 0).sum()
print(f'Cross-system days gap: sessions > impressions on {pos_gap:,}/{TOTAL_ROWS:,} rows ({pos_gap/TOTAL_ROWS*100:.1f}%) — direct traffic / referral / AI sessions, not a bug.')
print(f'Cross-system days gap: sessions = impressions on {zero_gap:,}/{TOTAL_ROWS:,} rows ({zero_gap/TOTAL_ROWS*100:.1f}%)')
print(f'Cross-system days gap: sessions < impressions on {neg_gap:,}/{TOTAL_ROWS:,} rows ({neg_gap/TOTAL_ROWS*100:.1f}%) — GSC-impression-only days (user saw link but did not click, or GA4 not triggered)')
window_ok = (df['days_with_impressions'].max() <= 90) and (df['days_with_sessions'].max() <= 90)
print(f'Window range OK?       : {window_ok}')

# --- Block 4: FIELD BUCKETS ---
print('\n' + '=' * 70)
print('CHECK 4 / 5 — FIELD BUCKETS (mutually exclusive, fully assigned, match ml_utils)')
print('=' * 70)
CONTEXT = ['content_id','client_id']
LABEL_PROXY = [
    'trend_direction','trend_pct','is_declining_label',
    'impressions_last_30d','impressions_prev_30d',
    'clicks_last_30d','sessions_last_30d',
    'clicks_prev_30d','sessions_prev_30d',
]
FEATURES_NUM = list(MODEL_NUMERIC_FEATURES)
FEATURES_CAT = list(MODEL_CATEGORICAL_FEATURES)
EXCLUDED = [
    'provider_used','model_used','char_count_tier',
    'impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d',
    'engaged_sessions_90d','pageviews_90d','users_90d','scroll_events_90d',
    'age_tier_order','has_clicks','has_ai_sessions','measurable_opportunity',
]
all_bucketed = CONTEXT + LABEL_PROXY + FEATURES_NUM + FEATURES_CAT + EXCLUDED
from collections import Counter
cnt = Counter(all_bucketed)
dupes_bucket = [c for c,n in cnt.items() if n > 1]
bucket_set = set(all_bucketed)
univ_set = set(all_universe)
missing_cols = sorted(univ_set - bucket_set)
extra_cols = sorted(bucket_set - univ_set)
match_num = sorted(FEATURES_NUM) == sorted(MODEL_NUMERIC_FEATURES)
match_cat = sorted(FEATURES_CAT) == sorted(MODEL_CATEGORICAL_FEATURES)
print(f'Bucket sizes: context={len(CONTEXT)} label_proxy={len(LABEL_PROXY)} feat_num={len(FEATURES_NUM)} feat_cat={len(FEATURES_CAT)} excluded={len(EXCLUDED)}')
print(f'Sum bucketed / universe : {len(all_bucketed)} / {len(all_universe)}')
print(f'Duplicate assignments   : {dupes_bucket if dupes_bucket else "(none)"}')
print(f'Unassigned columns      : {missing_cols if missing_cols else "(none)"}')
print(f'Extra (not in universe) : {extra_cols if extra_cols else "(none)"}')
print(f'Feat NUM matches ml_utils: {match_num}  ({len(FEATURES_NUM)} cols)')
print(f'Feat CAT matches ml_utils: {match_cat}  ({len(FEATURES_CAT)} cols)')
buckets_ok = (not dupes_bucket) and (not missing_cols) and (not extra_cols) and match_num and match_cat
print(f'Buckets clean?          : {buckets_ok}')

# --- Block 5: MISSINGNESS — check it follows content_type (skill §24) ---
print('\n' + '=' * 70)
print('CHECK 5 / 5 — MISSINGNESS (patterned, not random: follows content_type)')
print('=' * 70)
target_cols = ['search_volume','word_count','competition','cpc','competition_level','main_intent','model_used','provider_used','scroll_rate']
overall = df[target_cols].isna().mean().rename('overall_%').mul(100).round(1)
by_ct = df.groupby('content_type')[target_cols].apply(lambda g: g.isna().mean()).mul(100).round(1)
missing_table = pd.concat([overall.rename('OVERALL'), by_ct.T], axis=1)
print('Missing % by column + content_type:')
print(missing_table.to_string())
print(f'\nPattern check: "feedly article" missing search_volume = {by_ct.loc["feedly article", "search_volume"]:.1f}%  (expect ~100 — no keyword metadata for this type)')
print(f'Pattern check: "keyword article" missing word_count   = {by_ct.loc["keyword article", "word_count"]:.1f}%  (expect ~28 — skill §24 warning)')
print(f'Pattern check: "comparison article" missing search_volume = {by_ct.loc["comparison article", "search_volume"]:.1f}%, missing word_count = {by_ct.loc["comparison article", "word_count"]:.1f}%  (expect 0/0 — complete for this type)')

# --- Final summary line ---
print('\n' + '=' * 70)
all_ok = grain_ok and (len(all_raw_cols)==44) and (len(prep_added)==8) and (len(all_universe)==52) and window_ok and buckets_ok
print(f'OVERALL CONTRACT VERIFICATION: {"PASS" if all_ok else "FAIL"} — 5/5 checks')
print('=' * 70)

CHECK 1 / 5 — GRAIN: one row = one content_id
Total rows               : 30,000  (expect 30,000)
Unique content_id        : 30,000  (expect 30,000)
Duplicate content_ids    : 0  (expect 0 — grain holds)
Unique client_id         : 32  (expect 32)
Grain holds?             : True

CHECK 2 / 5 — COUNTS (columns, rows, clients)
Raw CSV cols             : 44  (expect 44)
Prep-script added cols   : 8  (expect 8)
Total contract universe  : 52  (expect 52)
Pages per client — min=3, median=567, max=7,008

CHECK 3 / 5 — WINDOW RANGES (must fit 90-day semantics)
content_age_days     min=90   median=236   max=564
days_with_impressions min=1   median=81   max=88  (expect 0-90)
days_with_sessions    min=1   median=6   max=90  (expect 0-90)
Cross-system days gap: sessions > impressions on 1,282/30,000 rows (4.3%) — direct traffic / referral / AI sessions, not a bug.
Cross-system days gap: sessions = impressions on 975/30,000 rows (3.2%)
Cross-system days gap: sessions < impressions on 27,743/30,000 ro

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


Five limits of the starter CSV that no amount of feature engineering or modeling can fix.
Each is verified below with a query.

1. **Exact snapshot date is unknown.** The CSV is a single 90-day trailing export with no
   `export_date` or `report_date` column. So we cannot tell whether the window covers
   (e.g.) "Jan 1 – Mar 31, 2026" vs "Feb 15 – May 15, 2026", which means:
   - We cannot detect seasonal patterns (holidays, quarter-end, industry events).
   - We cannot line this window up against other exports or external events.
   - Forward-window validation (ML-05) requires a panel dataset (the warehouse).

2. **Panel is flattened to a snapshot — per-client history depth is invisible.**
   `dim_clients.gsc_data_start` in the warehouse ranges from <3 months to 17 months.
   In the snapshot we have 32 clients but no per-client "data age" metadata. A page from
   a 3-month-old client is observationally identical to one from a 17-month-old client,
   even though a page's own `content_age_days` ≥ 90 is enforced by the filter. A page's
   performance metrics therefore reflect how long the *client* has been tracked, not just
   how good the content is.

3. **Days 61–90 have no dedicated comparison column.** The CSV exposes `*_last_30d`
   (days 0–30) and `*_prev_30d` (days 31–60), but days 61–90 contribute only to the
   `*_90d` totals. So the label's trend window (last 30 vs prev 30) uses only ⅔ of the
   snapshot's data, and a steady decline that *started* in days 61–90 is invisible to the
   label definition — the snapshot alone can't tell you whether a decline is new or
   already underway.

4. **Cross-system measurement drift is real but unquantified.** GSC (impressions/position/
   CTR) and GA4 (sessions/engagement/scroll) measure with different attribution windows,
   timezone boundaries, and bot-filtering rules. The snapshot hides the drift behind
   pre-computed rates. We know from Block 3 above that `days_with_sessions` can exceed
   `days_with_impressions` (direct / referral / AI traffic). But we cannot *quantify* the
   per-page drift from the snapshot alone. The warehouse with daily fact rows lets you
   compare channel-split days; the snapshot doesn't.

5. **`provider_used` / `model_used` are excluded for good reason — but that information
   loss is real.** The contract correctly buckets these as Excluded (product-decision
   metadata, not user signals). The cost: if one provider's pages consistently decline
   at 60% vs another at 45%, no model here can surface that, and no feature can proxy it
   because the CSV deliberately does not expose client-level feature counts per provider.
   An analyst (not a model) must run a separate provider audit.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4 code: 5 queries proving each data limit claim
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Portable repo-root walk-up ---
NB_PATH = Path(os.path.abspath('')).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / 'data' / 'raw').is_dir() and (REPO_ROOT / 'scripts' / 'ml_utils.py').is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(f'Repo root not found from {NB_PATH}')
SCRIPTS_DIR = str(REPO_ROOT / 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
assert RAW_PATH.is_file(), f'Missing CSV: {RAW_PATH}'
df = pd.read_csv(RAW_PATH)
TOTAL = len(df)

# --- LIMIT 1: no date column anywhere ---
print('=' * 70)
print('LIMIT 1 / 5 — NO EXACT SNAPSHOT DATE')
print('=' * 70)
date_candidates = [c for c in df.columns if any(k in c.lower() for k in ['date','time','export','snapshot','report','created','updated']) and c != 'days_since_last_update' and not c.startswith('days_with_')]
print(f'Columns matching date/time keywords (excluding day-count metrics): {date_candidates if date_candidates else "(none)"}')
print(f'days_since_last_update measures content-staleness (from content creation/update meta), not export date.')
print(f'min(days_since_last_update) = {df["days_since_last_update"].min():.0f}, max = {df["days_since_last_update"].max():.0f}  — these are content ages in days, not a snapshot timestamp.')
print('→ No export_date, report_date, snapshot_date column exists in the 44 raw CSV columns.')

# --- LIMIT 2: per-client history depth is hidden (show client volume heterogeneity instead) ---
print('\n' + '=' * 70)
print('LIMIT 2 / 5 — PER-CLIENT HISTORY DEPTH IS INVISIBLE (but volume heterogeneity is a proxy)')
print('=' * 70)
client_pages = df['client_id'].value_counts()
client_median_age = df.groupby('client_id')['content_age_days'].median()
client_median_imps = df.groupby('client_id')['impressions_90d'].median()
hetero = pd.DataFrame({
    'pages_per_client': client_pages,
    'median_content_age_days': client_median_age.reindex(client_pages.index),
    'median_impressions_90d': client_median_imps.reindex(client_pages.index),
}).sort_values('pages_per_client', ascending=False)
print(f'32 clients — pages spread: min={hetero["pages_per_client"].min()}, median={hetero["pages_per_client"].median():.0f}, max={hetero["pages_per_client"].max()}')
print(f'32 clients — median content_age spread: min={hetero["median_content_age_days"].min():.0f}d, max={hetero["median_content_age_days"].max():.0f}d')
print(f'32 clients — median imps_90d spread: min={hetero["median_impressions_90d"].min():.0f}, max={hetero["median_impressions_90d"].max():.0f}')
print(f'Ratio (max/min pages per client): {hetero["pages_per_client"].max()/hetero["pages_per_client"].min():.1f}×  — but no gsc_data_start metadata to explain it.')
print('→ Without dim_clients.gsc_data_start (warehouse only), we cannot tell if a low-page client is new or small.')

# --- LIMIT 3: days 61-90 have no dedicated 30-day column ---
print('\n' + '=' * 70)
print('LIMIT 3 / 5 — DAYS 61–90 ARE NOT A DEDICATED COMPARISON WINDOW')
print('=' * 70)
cols_30 = [c for c in df.columns if c.endswith('_30d')]
print(f'30-day sub-window columns: {cols_30}  ({len(cols_30)} total, = 3 metrics × last+prev = 6)')
print(f'No "*_days61_90" or "*_first_30d" column exists.')
# Quantify: for rows where prev_30d is non-zero, show how much of 90-day total lives in the unmeasured band
imps_day61_90 = df['impressions_90d'] - df['impressions_last_30d'] - df['impressions_prev_30d']
rows_prev_pos = df['impressions_prev_30d'] > 0
day61_share = (imps_day61_90[rows_prev_pos] / df.loc[rows_prev_pos, 'impressions_90d']).mul(100)
print(f'\nRows with prev_30d imps > 0: {int(rows_prev_pos.sum()):,}/{TOTAL:,} ({rows_prev_pos.mean()*100:.1f}%)')
print(f'Among those rows, share of 90-day imps coming from days 61–90 (unmeasured band):')
print(f'  min={day61_share.min():.1f}%   median={day61_share.median():.1f}%   p90={day61_share.quantile(0.9):.1f}%   max={day61_share.max():.1f}%')
print(f'  Rows where days 61–90 carry ≥ 1/3 of 90-day volume: {int((day61_share >= 33.3).sum()):,}/{int(rows_prev_pos.sum()):,}  ({(day61_share >= 33.3).mean()*100:.1f}%)')
print('→ The label (last-30 vs prev-30) is blind to a decline that began in days 61–90.')

# --- LIMIT 4: cross-system drift unquantified ---
print('\n' + '=' * 70)
print('LIMIT 4 / 5 — CROSS-SYSTEM DRIFT (GSC vs GA4) IS REAL BUT UNQUANTIFIABLE')
print('=' * 70)
# Click/session ratio: expected > 1 per channel but snapshot hides which channel a day is
gsc_clicks = df['clicks_90d'].sum()
ga4_sessions = df['sessions_90d'].sum()
ratio = gsc_clicks / ga4_sessions if ga4_sessions else np.nan
print(f'Total GSC clicks (90d)      : {gsc_clicks:,.0f}')
print(f'Total GA4 sessions (90d)    : {ga4_sessions:,.0f}')
print(f'Global clicks / sessions    : {ratio:.2f}  (naïve expectation: ≤ 1 click per session for pure search)')
# Per-page drift: rows with clicks > sessions (should be rare if same channel)
page_ratio = df['clicks_90d'] / df['sessions_90d'].replace(0, np.nan)
n_gt = int((page_ratio > 1).sum())
print(f'Pages with clicks_90d > sessions_90d: {n_gt:,}/{TOTAL:,} ({n_gt/TOTAL*100:.1f}%)')
print('  Interpretation: same user clicks → multi-session GA4 attribution, or non-search channels inflate sessions denominator.')
print('→ No per-day rows (warehouse only) → cannot split sessions by channel or quantify drift per-page.')

# --- LIMIT 5: provider/model info loss ---
print('\n' + '=' * 70)
print('LIMIT 5 / 5 — PROVIDER/MODEL INFO EXCLUDED (real information loss)')
print('=' * 70)
prov_decline = df.copy()
prov_decline['is_declining'] = (prov_decline['trend_direction'] == 'down').astype(int)
# provider_used coverage + decline rate by provider
prov_cov = prov_decline['provider_used'].notna().mean() * 100
prov_tbl = prov_decline.groupby('provider_used').agg(
    N=('content_id','count'),
    decline_rate=('is_declining','mean'),
    median_imps=('impressions_90d','median'),
).sort_values('N', ascending=False).round(3)
prov_tbl['decline_rate'] = (prov_tbl['decline_rate'] * 100).round(1)
print(f'provider_used fill rate: {prov_cov:.1f}%  ({TOTAL - int(prov_decline["provider_used"].isna().sum()):,}/{TOTAL:,} rows)')
print(f'\nDecline rate by provider_used (this IS a product-decision confound, not a feature — excluded from model):')
print(prov_tbl.to_string())
model_cov = prov_decline['model_used'].notna().mean() * 100
print(f'\nmodel_used fill rate: {model_cov:.1f}%')
print('→ Correctly excluded (product metadata, not user signal). But any real provider-level gap in decline rate is invisible to every model built on this contract.')

print('\n' + '=' * 70)
print('5/5 limits verified with queries above.')
print('=' * 70)


LIMIT 1 / 5 — NO EXACT SNAPSHOT DATE
Columns matching date/time keywords (excluding day-count metrics): (none)
days_since_last_update measures content-staleness (from content creation/update meta), not export date.
min(days_since_last_update) = 1, max = 373  — these are content ages in days, not a snapshot timestamp.
→ No export_date, report_date, snapshot_date column exists in the 44 raw CSV columns.

LIMIT 2 / 5 — PER-CLIENT HISTORY DEPTH IS INVISIBLE (but volume heterogeneity is a proxy)
32 clients — pages spread: min=3, median=567, max=7008
32 clients — median content_age spread: min=91d, max=557d
32 clients — median imps_90d spread: min=1, max=6108
Ratio (max/min pages per client): 2336.0×  — but no gsc_data_start metadata to explain it.
→ Without dim_clients.gsc_data_start (warehouse only), we cannot tell if a low-page client is new or small.

LIMIT 3 / 5 — DAYS 61–90 ARE NOT A DEDICATED COMPARISON WINDOW
30-day sub-window columns: ['impressions_last_30d', 'clicks_last_30d', 'ses

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.